# A local lakehouse: Delta tables on MinIO with Krishiv

This notebook talks to the `krishiv` container over Spark Connect with the ordinary PySpark client
(`pyspark-client`: no JVM here). The data lives in the `lakehouse` bucket of the `minio` container.

Run the cells top to bottom. Results are shown with `toPandas()`.

## Connect

The container sets `SPARK_REMOTE` (the server's address and token), so the builder needs no arguments.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sql("SELECT version() AS krishiv").toPandas()

## Write a Delta table to the bucket

In [ ]:
path = "s3://lakehouse/trips"

trips = spark.sql("""
    SELECT * FROM VALUES
        (1, 'Pune', 4.2D, 96.0D),
        (2, 'Pune', 11.8D, 240.0D),
        (3, 'Mumbai', 7.5D, 210.0D),
        (4, 'Nashik', 3.1D, 70.0D),
        (5, 'Mumbai', 19.4D, 520.0D)
    AS t(trip_id, city, distance_km, fare)
""")
trips.write.format("delta").mode("overwrite").save(path)

## Read it back

By path with the DataFrame reader, or by name once the table is in the catalog.

In [ ]:
spark.read.format("delta").load(path).orderBy("trip_id").toPandas()

In [ ]:
spark.sql(f"CREATE TABLE IF NOT EXISTS trips USING delta LOCATION '{path}'")

spark.sql("""
    SELECT city, count(*) AS trips, round(sum(fare), 2) AS fares
    FROM trips
    GROUP BY city
    ORDER BY city
""").toPandas()

## Change it with SQL

Each statement is one commit in the table's `_delta_log/`.

In [ ]:
spark.sql("INSERT INTO trips VALUES (6, 'Nashik', 8.9, 185.0)")
spark.sql("UPDATE trips SET fare = round(fare * 1.1, 2) WHERE city = 'Pune'")
spark.sql("DELETE FROM trips WHERE distance_km < 4")

spark.table("trips").orderBy("trip_id").toPandas()

## Check

The cells above did what they say (this is also what CI runs).

In [ ]:
now = {r.trip_id: r.fare for r in spark.read.format("delta").load(path).collect()}
assert now == {1: 105.6, 2: 264.0, 3: 210.0, 5: 520.0, 6: 185.0}, now
print("ok:", len(now), "trips")

## Next

- The Krishiv console at `https://localhost:8080/` lists these queries (token `krishiv-quickstart-token-change-me-please`).
- The MinIO console at <http://127.0.0.1:9001> shows the table's files in the `lakehouse` bucket.
- Your own PySpark scripts connect the same way: see the README next to `compose.yaml`.